# NLA Reliability — Analysis Synthesis

This notebook summarizes **reliability** (does the AV→AR round-trip agree with itself?) and **validity** (does the result still encode the label we care about?) across four canonical runs.

| Run ID | Data path | What Gemma sees at Step 1 |
|--------|-----------|-------------------------|
| **prism** | `data/runs/prism/` | Last user turn of a multi-turn chat |
| **biosbias** | `data/runs/biosbias/` | Full biography text (gender scrubbed) |
| **mmlu_choice** | `data/runs/mmlu_choice/` | `Question: …` + A–D options + `Answer:` |
| **mmlu_nochoice** | `data/runs/mmlu_nochoice/` | Question stem only (no choices in prompt) |

**Design:** 400 items × 12 stochastic AV samples (T=1.0) → 4,800 reconstructions per run. Gemma-3-12B **layer 32**, L2-normalized activations, Anthropic NLA checkpoints `kitft/nla-gemma3-12b-L32-{av,ar}`.

**Refresh CSV inputs (run after `pull_from_modal.py --all-runs`):**
```bash
uv run python scripts/build_synthesis_tables.py
```

Includes centered cosines, text consistency, linear probes, and **G-theory** (G-study / D-study on `fidelity_cos` and per-sample **within-item `consistency_cos`**).

## How to read cosine metrics

### Raw cosines are inflated
All Step-1 activations share almost the same direction: **‖μ‖ ≈ 0.99** (mean activation norm). So **any two** vectors have raw cosine ≈ **0.98–0.99**, even for unrelated items. A matched fidelity of 0.99 does **not** by itself mean excellent reconstruction.

### Mean-centering
Subtract the global mean activation **μ** from every vector, re-normalize, then compute cosine. This removes the shared background direction and exposes **item-specific** structure.

### Fidelity (original vs reconstruction)
- **Matched:** cosine between activation *i* and its own reconstruction(s).
- **Mismatched:** cosine between activation *j* (j≠i) and reconstruction *i* (random wrong activations).
- **Gap = matched − mismatched.** Large gap → reconstructions align to the **correct** activation more than to wrong ones.

### Consistency (reconstruction vs reconstruction)
- **Within-item:** cosine between two of the 12 recons for the **same** activation.
- **Between-item:** cosine between recons from **different** activations.
- **Gap = within − between.** Large gap → 12 verbalizations for one activation agree with each other more than with other items' recons.

**Reliability** is about gaps *and* the level of centered similarities (e.g. high within-item + low between-item).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
REPORTS = ROOT / "reports"

RUN_ORDER = ["prism", "biosbias", "mmlu_choice", "mmlu_nochoice"]
RUN_LABELS = {
    "prism": "PRISM",
    "biosbias": "Bias in Bios",
    "mmlu_choice": "MMLU (+ choices)",
    "mmlu_nochoice": "MMLU (Q only)",
}

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.style.use("seaborn-v0_8-whitegrid")

headline = pd.read_csv(REPORTS / "synthesis_headline_metrics.csv")
inventory = pd.read_csv(REPORTS / "synthesis_inventory.csv")
text_df = pd.read_csv(REPORTS / "synthesis_text_consistency.csv")
probes = pd.read_csv(REPORTS / "synthesis_linear_probes.csv")
gvar = pd.read_csv(REPORTS / "synthesis_g_theory_variance.csv")
gdstudy = pd.read_csv(REPORTS / "synthesis_g_theory_d_study.csv")
if "metric" not in gvar.columns:
    gvar["metric"] = "fidelity_cos"
    gdstudy["metric"] = "fidelity_cos"


def pivot_metrics(metrics: list[str], stat: str = "mean") -> pd.DataFrame:
    sub = headline[headline["metric"].isin(metrics)].copy()
    p = sub.pivot(index="metric", columns="run_id", values=stat)
    cols = [c for c in RUN_ORDER if c in p.columns]
    p = p.reindex(metrics)[cols]
    p.columns = [RUN_LABELS[c] for c in p.columns]
    return p

## 1. Data inventory

In [ ]:
display(inventory)

## 2. Mean-centered cosine similarities (vector space)

Tables report **mean** cosine (and gaps) over all sampled pairs. Std / percentiles are in `reports/synthesis_headline_metrics.csv`.

In [ ]:
FIDELITY = [
    "Fidelity centered (matched)",
    "Fidelity centered (mismatched)",
    "Fidelity centered gap",
]
CONSISTENCY = [
    "Consistency centered (within-item)",
    "Consistency centered (between-item)",
    "Consistency centered gap",
]

display(Markdown("### 2a. Fidelity — original vs reconstruction"))
display(pivot_metrics(FIDELITY))

display(Markdown("### 2b. Consistency — reconstruction vs reconstruction"))
display(pivot_metrics(CONSISTENCY))

display(Markdown("### 2c. Cosine inflation diagnostic"))
display(pivot_metrics(["||mean activation||"]))

### 2d. How to read these numbers

**PRISM & Bias in Bios**
- **Fidelity matched** ~0.65–0.67: after centering, reconstructions still point somewhat toward their own activation.
- **Fidelity mismatched** ~0: wrong activations are not aligned with the recon → **gap ~0.65–0.67**.
- **Consistency within** ~0.95: 12 recons for one item strongly agree.
- **Consistency between** ~0.15: recons from different items are nearly orthogonal after centering → **gap ~0.79–0.81**.
- **Takeaway:** Strong **reliability** in vector space — the round-trip is stable and activation-specific.

**MMLU (with choices in prompt)**
- **Fidelity matched** only ~0.24; **mismatched** also ~0 → small **fidelity gap** (~0.25).
- **Consistency within** ~0.92 (still high) but **between** ~0.65 (much higher than PRISM) → **gap only ~0.27**.
- Many items from the same MMLU subject look similar after recon; the model does not separate items as cleanly.
- **Takeaway:** High raw cosines would look "fine"; centered metrics show **weak item specificity**.

**MMLU (question only)**
- **Fidelity matched** ~0.59, **gap** ~0.59 — better than MCQ format.
- **Consistency within** ~0.93, **between** ~0.21, **gap** ~0.72 — closer to PRISM/BiasBios than MCQ MMLU.
- Removing choices from the prompt improves centered reliability, but probes still drop (see §4).

**Pattern:** Large **within** + small **between** + large **gaps** = good reliability. MMLU+choices fails mainly on **between** staying high.

## 3. Text-space consistency (MPNet)

Do the 12 AV **wordings** agree in semantic space? Embed each description with `all-mpnet-base-v2`, average pairwise cosine per activation.

This is usually **much lower** than recon cosine (~0.84 vs ~0.999 raw) — language varies more than reconstructed directions.

In [ ]:
text_show = text_df.copy()
text_show["run"] = text_show["run_id"].map(RUN_LABELS)
display(text_show.set_index("run")[["mean", "std", "median", "p5", "p95"]])

- **PRISM / BiasBios ~0.86–0.87:** moderate paraphrase consistency across 12 stochastic AV samples.
- **MMLU question-only ~0.84:** slightly lower than PRISM/BiasBios; paraphrases of bare stems still agree fairly well in MPNet space.
- **MMLU with choices ~0.83:** a bit lower than question-only (longer prompts with A–D add surface variation across the 12 AV samples).
- Text consistency does **not** predict recon consistency well — AR compresses diverse wordings.

## 4. Linear probes (validity)

Logistic regression on **original** activations vs **mean of 12 recon vectors** (same 80/20 split, seed 42). Tests whether metadata labels remain linearly decodable — a coarse validity check, not full semantic equality.

In [ ]:
probe_show = probes.copy()
probe_show["run"] = probe_show["run_id"].map(RUN_LABELS)
display(probe_show)

pivot_probe = probes.pivot_table(
    index=["run_id", "target"],
    columns="vector_source",
    values="probe_acc",
)
pivot_probe["delta (recon − orig)"] = pivot_probe["recon_mean"] - pivot_probe["original"]
pivot_probe.index = pd.MultiIndex.from_tuples(
    [(RUN_LABELS.get(r, r), t) for r, t in pivot_probe.index],
    names=["run", "target"],
)
display(Markdown("### Probe accuracy (test set)"))
display(pivot_probe)

### 4b. Validity interpretation

| Run | Label | Original | Mean recon | Δ | Reading |
|-----|-------|----------|------------|---|--------|
| PRISM | gender | 0.58 | 0.58 | 0.00 | Round-trip preserves coarse gender signal (class imbalance limits probe) |
| BiasBios | profession | 0.30 | 0.26 | −0.04 | Slight loss on 26-class profession |
| BiasBios | gender | 0.55 | 0.59 | +0.04 | Small gain on binary gender |
| MMLU (+ choices) | subject | **0.96** | **0.76** | **−0.20** | Strong subject code in activations; large drop after recon |
| MMLU (Q only) | subject | **0.98** | **0.94** | **−0.04** | Better preservation when prompt is question-only |

**Key point:** MMLU can show **high recon reliability** (within-item consistency) while **losing subject decodability** after round-trip — especially with MCQ-formatted prompts. That is **reliability ≠ validity**.

## 5. Generalizability theory (G-study / D-study)

Same **p × i** design (400 activations × 12 AV samples), two scores per cell:

| Score | Source | Cell value |
|-------|--------|------------|
| **`fidelity_cos`** | `fidelity_scores_*.parquet` | cos(original, recon) for that sample |
| **`consistency_cos`** | `pairwise_consistency_*.parquet` | mean within-item recon–recon cos for that sample vs the other 11 |

| Facet | Meaning |
|-------|--------|
| **p** (400) | Which activation / item |
| **i** (12) | Which stochastic AV→AR sample |

**Variance components:** σ²_p (between activations), σ²_i (global sample facet), σ²_pi (activation×sample instability).

**n′ in the D-study** = how many of the 12 samples you **average** per activation (1 = single draw; 12 = full pipeline). **G_rel(n′)** = dependability for **comparing activations** on that score.

In [ ]:
def show_g_theory(metric: str, section: str) -> None:
    """Variance table + D-study G_rel curves for one G-study score."""
    gv = gvar[gvar["metric"] == metric].copy()
    gd = gdstudy[gdstudy["metric"] == metric].copy()
    if gv.empty:
        display(Markdown(f"*{section}: no rows for `{metric}` — run `build_synthesis_tables.py`*"))
        return
    gv["run"] = gv["run_id"].map(RUN_LABELS)
    display(Markdown(f"### {section}a. Variance components (%) — `{metric}`"))
    display(
        gv[["run", "var_pct_p", "var_pct_pi", "var_pct_i", "cronbach_alpha", "G_rel_n1", "G_rel_n12"]]
        .set_index("run")
    )
    display(Markdown(f"### {section}b. D-study — G_rel(n′) — `{metric}`"))
    for rid in RUN_ORDER:
        if rid not in gd["run_id"].values:
            continue
        sub = gd[gd["run_id"] == rid][["n_samples", "G_rel"]].copy()
        sub.columns = ["n′", "G_rel"]
        display(Markdown(f"**{RUN_LABELS[rid]}**"))
        display(sub.set_index("n′"))


show_g_theory("fidelity_cos", "5")
show_g_theory("consistency_cos", "5′")

### 5c. Interpretation

**Fidelity (`fidelity_cos`)** — stability of “does this recon match its own activation?”

- **PRISM / BiasBios:** ~91–99% σ²_p; **G_rel(n′=12) ≈ 0.99** — dependable per-activation fidelity means.
- **MMLU + choices:** ~75% σ²_p, ~24% σ²_pi; **G_rel(n′=1) ≈ 0.76** — one verbalization is a shaky fidelity estimate; n′=12 → ~0.97.
- **MMLU question only:** ~91% σ²_p — closer to PRISM; aligns with better centered gaps and probes.

**Consistency (`consistency_cos`)** — stability of “do the 12 recons for this item agree?” (per-sample mean within-item recon cosine)

- Expect **lower σ²_p** than fidelity when all items have similarly high raw within-item cosines; **σ²_pi** then captures item-specific verbalization spread.
- Compare **5 vs 5′** side by side: MMLU + choices should show more consistency sample noise (higher var_pct_pi, lower G_rel at n′=1) if MCQ prompts hurt recon agreement.

CLI:

```bash
uv run python scripts/g_theory_study.py --run-id mmlu_choice --metric all
uv run python scripts/g_theory_study.py --run-id mmlu_nochoice --metric all
uv run python scripts/build_synthesis_tables.py
```

## 6. MMLU prompt format comparison

Same items and pipeline; only Step-1 `prompt_text` differs (MCQ vs question stem).

In [ ]:
mmlu_cols = [RUN_LABELS["mmlu_choice"], RUN_LABELS["mmlu_nochoice"]]

display(Markdown("### Centered fidelity"))
display(pivot_metrics(FIDELITY).loc[:, mmlu_cols])

display(Markdown("### Centered consistency"))
display(pivot_metrics(CONSISTENCY).loc[:, mmlu_cols])

display(Markdown("### Subject probe"))
mmlu_probe = probes[probes["run_id"].str.startswith("mmlu") & (probes["target"] == "subject")]
display(mmlu_probe.pivot_table(index="run_id", columns="vector_source", values="probe_acc").rename(index=RUN_LABELS))

**MCQ → question-only changes:**
- Fidelity **matched** 0.24 → 0.59; fidelity **gap** 0.25 → 0.59.
- Consistency **between** 0.65 → 0.21; consistency **gap** 0.27 → 0.72.
- Subject probe drop 0.96→0.76 becomes 0.98→0.94.

Including answer choices in the prompt pulls activations toward a **shared exam-like format**, so reconstructions are less item-specific. Question-only prompts improve both **centered reliability** and **probe validity**, but MMLU still shows a validity drop relative to originals.

## 7. Summary figures — centered cosines and gaps

In [ ]:
gap_only = pivot_metrics(["Fidelity centered gap", "Consistency centered gap"])
ax = gap_only.T.plot(kind="bar", figsize=(11, 4), rot=0)
ax.set_ylabel("Mean centered cosine gap")
ax.set_title("Reliability gaps (higher = more activation-specific)")
ax.axhline(0, color="k", linewidth=0.8)
plt.legend(title="", loc="best")
plt.tight_layout()
plt.show()

within_between = pivot_metrics([
    "Consistency centered (within-item)",
    "Consistency centered (between-item)",
])
ax = within_between.T.plot(kind="bar", figsize=(11, 4), rot=0)
ax.set_ylabel("Mean centered cosine")
ax.set_title("Consistency: within-item vs between-item (centered)")
plt.tight_layout()
plt.show()

## 8. Optional follow-ups

- `scripts/mmlu_subject_consistency.py` — same-subject vs cross-subject between pairs for MMLU.
- `scripts/diagnose_reconstruction_fidelity.py` / `consistency.py` — full distribution output per dataset.
- `scripts/g_theory_study.py` — regenerate `reports/g_theory_*.csv` for a single dataset.
- Re-run `uv run python scripts/build_synthesis_tables.py` after pulling new artifacts.